# Tracelet crash detection — train an on-device crash model

Supports [Tracelet issue #183](https://github.com/Ikolvi/Tracelet/issues/183): train a
phone-IMU **crash classifier** whose features mirror the signals the Tracelet SDK already
has at inference time, then export an artifact + a recommended threshold tuning for the
rule-based detector.

**Dataset:** [Kaggle — Smartphone IMU Road Accident Detection](https://www.kaggle.com/datasets/drabdulbari/smartphone-imu-road-accident-detection-dataset)
(`accel XYZ + gyro XYZ + speed + GPS + Crash_Label`, ~8,000 rows). License: **CC0** — public
domain, so a model trained on it is commercially shippable. (This notebook deliberately sets
aside the VZCrash CC BY-NC licensing block discussed in #183; CC0 unblocks training/shipping.)

**Why these features.** The SDK's detector (`sdk/rust-core/core/src/algorithms/impact.rs`)
classifies each impact window from three signals only:

| SDK signal | Meaning | Rule default |
|---|---|---|
| `peak_g` | peak accel magnitude (g) | `crash_g_threshold = 2.0` |
| `speed_before_mps` | pre-impact speed | `crash_min_speed_kmh = 25` |
| `gyro_peak_dps` | peak rotation (deg/s) | corroboration `>= 100`, relax x0.7 |

So we engineer exactly `[accel_g, gyro_dps, speed_kmh]` per sample. A model trained on the same
features the SDK can compute on-device is directly portable, and its decision boundary can be
distilled back into `ImpactConfig` thresholds.

**Outputs:** trained RandomForest + GradientBoosting, metrics vs. the current rule baseline, an
ONNX export, a portable JSON tree dump, and a recommended `ImpactConfig` tuning.

> Caveat (from #183): the only CC0 set is ~8k *simulated* rows. Treat the resulting model as an
> **experimental baseline**, not a shippable production crash classifier — the production path is
> first-party field data (#173). This notebook is the reproducible training pipeline for when a
> larger, real, permissively-licensed (or licensed) dataset is available.

In [ ]:
!pip -q install kagglehub pandas numpy scikit-learn matplotlib joblib skl2onnx onnxruntime

## 1. Download the dataset

`kagglehub` pulls public datasets without manual Kaggle API keys in Colab. If it prompts for
auth, accept the dataset terms once on the Kaggle page, or drop a `kaggle.json` in `~/.kaggle/`.

In [ ]:
import glob, os
import pandas as pd, numpy as np

DATASET = 'drabdulbari/smartphone-imu-road-accident-detection-dataset'

csv_path = None
try:
    import kagglehub
    path = kagglehub.dataset_download(DATASET)
    print('Downloaded to:', path)
    csvs = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
    print('CSV files:', csvs)
    csv_path = csvs[0] if csvs else None
except Exception as e:
    print('kagglehub download failed:', e)
    print('Fallback: upload the CSV manually, then set csv_path = "/content/your.csv"')

assert csv_path, 'No CSV resolved — set csv_path manually before continuing.'
df = pd.read_csv(csv_path)
print('shape:', df.shape)
df.head()

## 2. Identify columns (robust to naming)

Different mirrors of this dataset use slightly different headers (`AccX`/`acc_x`/`ax`, …). We
detect accel XYZ, gyro XYZ, speed, and the crash label heuristically and print what we found —
**verify the mapping before training.**

In [ ]:
import re
cols = list(df.columns)
lc = {c: c.lower() for c in cols}

def pick(patterns, exclude=()):
    for c in cols:
        n = lc[c]
        if any(re.search(p, n) for p in patterns) and not any(re.search(e, n) for e in exclude):
            return c
    return None

ax = pick([r'acc.*x', r'\bax\b', r'a_x'])
ay = pick([r'acc.*y', r'\bay\b', r'a_y'])
az = pick([r'acc.*z', r'\baz\b', r'a_z'])
gx = pick([r'gyr.*x', r'\bgx\b', r'g_x'])
gy = pick([r'gyr.*y', r'\bgy\b', r'g_y'])
gz = pick([r'gyr.*z', r'\bgz\b', r'g_z'])
speed_col = pick([r'speed', r'velocit', r'kmh', r'kph', r'\bspd\b'])
label_col = pick([r'crash', r'accident', r'label', r'target', r'\by\b'])

mapping = dict(accel=(ax, ay, az), gyro=(gx, gy, gz), speed=speed_col, label=label_col)
print('Detected mapping:')
for k, v in mapping.items():
    print(f'  {k:6s}: {v}')

assert all([ax, ay, az, label_col]), 'Could not detect accel XYZ + label — set columns manually.'

## 3. Feature engineering — mirror the SDK's signals

- **accel magnitude → g**: `sqrt(ax^2+ay^2+az^2)`. Auto-detect whether the data is in m/s^2 or
  already in g (by typical resting magnitude ~9.81 vs ~1.0) and convert to **g**.
- **gyro magnitude → deg/s**: `sqrt(gx^2+gy^2+gz^2)`, converting from rad/s if needed.
- **speed → km/h**: detect m/s vs km/h by range and normalise.

This produces the exact `[accel_g, gyro_dps, speed_kmh]` triple the SDK evaluates per window.

In [ ]:
a = np.sqrt(df[ax].astype(float)**2 + df[ay].astype(float)**2 + df[az].astype(float)**2)
# Resting accel magnitude tells us the unit: ~9.81 => m/s^2, ~1.0 => g.
med_a = np.nanmedian(a)
accel_g = a / 9.80665 if med_a > 3.0 else a
print(f'accel median raw={med_a:.2f} -> treating as {"m/s^2" if med_a > 3.0 else "g"}; accel_g median={np.nanmedian(accel_g):.2f}')

if gx and gy and gz:
    g = np.sqrt(df[gx].astype(float)**2 + df[gy].astype(float)**2 + df[gz].astype(float)**2)
    # rad/s peaks are O(1-10); deg/s peaks O(100s). If 95th pct < 50 assume rad/s.
    gyro_dps = np.degrees(g) if np.nanpercentile(g, 95) < 50 else g
else:
    gyro_dps = pd.Series(np.zeros(len(df)))  # dataset without gyro -> 0 (no corroboration)
print(f'gyro_dps median={np.nanmedian(gyro_dps):.2f}, p95={np.nanpercentile(gyro_dps,95):.2f}')

if speed_col:
    s = df[speed_col].astype(float)
    speed_kmh = s * 3.6 if np.nanpercentile(s, 95) < 60 else s  # m/s if small range
else:
    speed_kmh = pd.Series(np.full(len(df), np.nan))
print(f'speed_kmh median={np.nanmedian(speed_kmh):.2f}, p95={np.nanpercentile(speed_kmh,95):.2f}')

y = df[label_col].astype(float).round().astype(int)
print('label balance:', y.value_counts(normalize=True).to_dict())

feat = pd.DataFrame({'accel_g': accel_g.values, 'gyro_dps': np.asarray(gyro_dps),
                     'speed_kmh': np.asarray(speed_kmh)})
# speed may be missing in some mirrors; keep it but median-fill so models still train.
feat['speed_kmh'] = feat['speed_kmh'].fillna(feat['speed_kmh'].median())
feat = feat.fillna(0.0)
X = feat[['accel_g', 'gyro_dps', 'speed_kmh']].values
feat.describe()

## 4. Rule-based baseline (replicates `impact.rs`)

The current SDK rule: **crash** if `accel_g >= 2.0` **and** `speed_kmh >= 25`, with gyro
corroboration relaxing the g-threshold by x0.7 when `gyro_dps >= 100`. We score it so the ML
models have an honest reference to beat.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

CRASH_G = 2.0; MIN_SPEED_KMH = 25.0; GYRO_DPS = 100.0; GYRO_RELAX = 0.7

def rule_predict(F):
    thr = np.where(F['gyro_dps'].values >= GYRO_DPS, CRASH_G * GYRO_RELAX, CRASH_G)
    return ((F['accel_g'].values >= thr) & (F['speed_kmh'].values >= MIN_SPEED_KMH)).astype(int)

y_rule = rule_predict(feat)
print('=== Rule baseline (impact.rs) ===')
print(classification_report(y, y_rule, digits=3))
print('confusion [tn fp / fn tp]:\n', confusion_matrix(y, y_rule))

## 5. Train ML models

Stratified split, `class_weight='balanced'` for the crash imbalance. RandomForest (the model
#183 cites as showing ~2x recall potential) plus GradientBoosting for comparison.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

Xtr, Xte, ytr, yte, Ftr, Fte = train_test_split(
    X, y, feat, test_size=0.25, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced',
                            random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr)

gb = GradientBoostingClassifier(random_state=42)
gb.fit(Xtr, ytr)
print('Trained RandomForest + GradientBoosting on', Xtr.shape[0], 'samples.')

## 6. Evaluate vs. the rule baseline

Crash detection favours **recall** (a missed crash is worse than a cancellable false alarm — the
SDK shows a cancel-countdown). We report precision/recall/F1 + PR-AUC and compare on the same
held-out split.

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             average_precision_score, roc_auc_score)

for name, model in [('RandomForest', rf), ('GradientBoosting', gb)]:
    p = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]
    print(f'=== {name} ===')
    print(classification_report(yte, p, digits=3))
    print('PR-AUC :', round(average_precision_score(yte, proba), 4),
          '| ROC-AUC:', round(roc_auc_score(yte, proba), 4))
    print('confusion [tn fp / fn tp]:\n', confusion_matrix(yte, p))
    print()

print('=== Rule baseline on the SAME test split ===')
yte_rule = rule_predict(Fte)
print(classification_report(yte, yte_rule, digits=3))
print('confusion:\n', confusion_matrix(yte, yte_rule))

## 7. Feature importance

In [ ]:
import matplotlib.pyplot as plt
names = ['accel_g', 'gyro_dps', 'speed_kmh']
imp = rf.feature_importances_
plt.figure(figsize=(5, 3))
plt.barh(names, imp)
plt.title('RandomForest feature importance')
plt.tight_layout(); plt.show()
print(dict(zip(names, np.round(imp, 3))))

## 8. Pick a recall-first operating point + distil thresholds

Two products from the model:
1. A **probability threshold** on the RF for a target recall (default 0.95), with its precision.
2. An **interpretable threshold tuning** for `ImpactConfig`: we fit a shallow decision tree on
   `[accel_g, speed_kmh]` and read off the split points, giving learned analogues of
   `crash_g_threshold` / `crash_min_speed_kmh` that you can drop into the rule engine.

In [ ]:
from sklearn.metrics import precision_recall_curve
from sklearn.tree import DecisionTreeClassifier, export_text

proba = rf.predict_proba(Xte)[:, 1]
prec, rec, thr = precision_recall_curve(yte, proba)
TARGET_RECALL = 0.95
ok = np.where(rec[:-1] >= TARGET_RECALL)[0]
if len(ok):
    i = ok[np.argmax(prec[ok])]
    print(f'For recall>={TARGET_RECALL}: prob_threshold={thr[i]:.3f}, precision={prec[i]:.3f}, recall={rec[i]:.3f}')
    RF_PROB_THRESHOLD = float(thr[i])
else:
    RF_PROB_THRESHOLD = 0.5
    print('Target recall not reachable; defaulting prob_threshold=0.5')

# Interpretable tuning for the rule engine (accel + speed only, like impact.rs).
dt = DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=42)
dt.fit(feat[['accel_g', 'speed_kmh']].values, y)
print('\nLearned decision tree (analogue of the rule thresholds):')
print(export_text(dt, feature_names=['accel_g', 'speed_kmh']))

## 9. Export artifacts

- `crash_rf.joblib` — the sklearn model (Python reuse).
- `crash_rf.onnx` — portable inference (ONNX Runtime / mobile).
- `crash_rf_trees.json` — self-contained tree dump for a future dependency-free on-device
  evaluator (mirrors how the Rust core avoids a heavy ML runtime).
- `impact_config_recommendation.json` — suggested `ImpactConfig` tuning + the chosen RF
  probability threshold, ready to feed back into `impact.rs`.

In [ ]:
import json, joblib

joblib.dump(rf, '/content/crash_rf.joblib')

# ONNX export
try:
    from skl2onnx import to_onnx
    onx = to_onnx(rf, Xtr.astype(np.float32))
    with open('/content/crash_rf.onnx', 'wb') as f:
        f.write(onx.SerializeToString())
    print('Wrote crash_rf.onnx')
except Exception as e:
    print('ONNX export skipped:', e)

# Portable tree dump (each tree as thresholds/children/values).
def tree_to_dict(t):
    tt = t.tree_
    return {
        'feature': tt.feature.tolist(),
        'threshold': tt.threshold.tolist(),
        'children_left': tt.children_left.tolist(),
        'children_right': tt.children_right.tolist(),
        'value': [v[0].tolist() for v in tt.value],
    }
forest = {'features': ['accel_g', 'gyro_dps', 'speed_kmh'],
          'classes': rf.classes_.tolist(),
          'trees': [tree_to_dict(e) for e in rf.estimators_]}
with open('/content/crash_rf_trees.json', 'w') as f:
    json.dump(forest, f)
print('Wrote crash_rf_trees.json (%d trees)' % len(forest['trees']))

recommendation = {
    'source_dataset': DATASET,
    'features': ['accel_g', 'gyro_dps', 'speed_kmh'],
    'rf_probability_threshold': RF_PROB_THRESHOLD,
    'current_rule': {'crash_g_threshold': CRASH_G, 'crash_min_speed_kmh': MIN_SPEED_KMH,
                     'gyro_corroboration_dps': GYRO_DPS, 'gyro_threshold_relax': GYRO_RELAX},
    'note': 'Experimental baseline on ~8k simulated CC0 rows (#183). Not for production; '
            'use first-party field data (#173) before promoting beta -> stable.'}
with open('/content/impact_config_recommendation.json', 'w') as f:
    json.dump(recommendation, f, indent=2)
print(json.dumps(recommendation, indent=2))

In [ ]:
# Bundle + download
import shutil
shutil.make_archive('/content/crash_model_artifacts', 'zip', '/content',
                    base_dir=None)
try:
    from google.colab import files
    for f in ['crash_rf.joblib', 'crash_rf.onnx', 'crash_rf_trees.json',
              'impact_config_recommendation.json']:
        if os.path.exists('/content/' + f):
            files.download('/content/' + f)
except Exception as e:
    print('Manual download from the Files pane. Artifacts in /content/.', e)

## 10. Integration notes

- **Mapping to the SDK.** The features (`accel_g`, `gyro_dps`, `speed_kmh`) are exactly the
  arguments to `ImpactDetector::on_impact_window` in `sdk/rust-core/core/src/algorithms/impact.rs`.
  The simplest, lowest-risk integration is to apply `impact_config_recommendation.json` as a
  **tuning of the existing rule** (update `crash_g_threshold` / `crash_min_speed_kmh`) rather than
  shipping a model runtime.
- **If you want the learned forest on-device,** `crash_rf_trees.json` is a dependency-free dump a
  small Rust evaluator can walk — no TFLite/ONNX runtime needed, matching the core's no-heavy-deps
  design. `crash_rf.onnx` is the alternative if you add an ONNX runtime.
- **Do not auto-promote beta -> stable** off this. Per #183 the CC0 set is ~8k *simulated* rows;
  this is a reproducible baseline. Production needs real first-party field data (#173) and field
  validation; in parallel, request research access to the real VZCrash / TITS-2021 sets.